<div style="color:#3c4d5a; border-top: 7px solid #42A5F5; border-bottom: 7px solid #42A5F5; padding: 5px; text-align: center; text-transform: uppercase"><h1>Práctica 3: Preparación de Datos y KNN - Función de Predicción</h1> </div> 

Desarrollado por: Domenica Beltran y Mario Reinoso

- [Enunciado de la práctica](#enunciado)

- [1. Cargar el dataset](#dataset)

- [2. Identificar los tipos de variables](#variables)

- [3. Realizar transformación categórica](transform)

- [4. Aplicar estandarización](#std)

- [5. Implementar KNN con distancia euclídea y k=3](#knn)

- [Ejemplo de prueba](#ejemplo)

- [Conclusiones](#conclusiones)

- [Referencias](#referencias)

<div id="enunciado" style="color:#37475a; border-bottom: 7px solid orange; width: 100%; margin-bottom: 15px; padding-bottom: 2px"><h2>Enunciado de la práctica</h2> </div>

Se dispone de un dataset en formato CSV con información básica de pacientes ecuatorianos, incluyendo variables demográficas y de salud. El objetivo es aplicar un flujo básico de Machine Learning utilizando el algoritmo K-Nearest Neighbors (KNN) para predecir si un nuevo paciente tiene o no diabetes.

Dataset:<br>

sexo,ciudad,colesterol,edad,diabetes <br>
1,Cuenca,bajo,18,no <br>
2,Quito,alto,52,si <br>
2,Guayaquil,medio,34,no <br>
1,Loja,alto,61,si <br>
2,Ambato,medio,45,no <br>
1,Machala,muy alto,67,si <br>

<div id="Importacion" style="color:#106ba3"><h3>Importación de librerías</h3> </div>

In [128]:
import pandas as pd
import copy
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
import numpy as np
print('Módulos importados')


Módulos importados


<div id="dataset" style="color:#37475a; border-bottom: 7px solid orange; width: 100%; margin-bottom: 15px; padding-bottom: 2px"><h2>1. Cargar el dataset</h2> </div>

### Nombre del dataset: Dataset de Diabetes en Pacientes Ecuatorianos

Enlace: [Disponible en formato CSV local / No especificado]

Descripción general: Este conjunto de datos clasifica a pacientes ecuatorianos mediante variables demográficas y de salud con el objetivo 
de predecir la presencia o ausencia de diabetes utilizando el algoritmo K-Nearest Neighbors (KNN).

Número de Variables (o atributos): 5 (sexo, ciudad, colesterol, edad, diabetes)

Número de instancias (pacientes): 6 (en la muestra proporcionada)

Salida: diabetes (NO: no tiene diabetes, SI: tiene diabetes)

In [129]:
nombresVariables = ['sexo', 'ciudad', 'colesterol', 'edad', 'diabetes']

# Cargar dataframe de un archivo local (Formato CSV)
dataset = "data.csv"
dfOriginal = pd.read_csv(dataset, sep=',', names=nombresVariables, header=0)

print('cantidad de observaciones (pacientes): ', dfOriginal.shape[0])
print('cantidad de variables: ', dfOriginal.shape[1])
print(dfOriginal.shape)
dfOriginal.head(6)

cantidad de observaciones (pacientes):  6
cantidad de variables:  5
(6, 5)


,sexo,ciudad,colesterol,edad,diabetes
0,1,Cuenca,bajo,18,no
1,2,Quito,alto,52,si
2,2,Guayaquil,medio,34,no
3,1,Loja,alto,61,si
4,2,Ambato,medio,45,no
5,1,Machala,muy alto,67,si


<div id="variables" style="color:#37475a; border-bottom: 7px solid orange; width: 100%; margin-bottom: 15px; padding-bottom: 2px"><h2>2. Identificar los tipos de variables</h2> </div>

In [130]:
# Creación de una copia profunda para preservar el dataframe original
dataframe = copy.deepcopy(dfOriginal)

# Aislamiento de la variable objetivo (target)
Y = dataframe['diabetes']

# Eliminación del target para conservar únicamente las variables predictoras
dataframe = dataframe.drop(['diabetes'], axis=1)

# Verificación de que la columna 'diabetes' ya no está presente
dataframe.head()

,sexo,ciudad,colesterol,edad
0,1,Cuenca,bajo,18
1,2,Quito,alto,52
2,2,Guayaquil,medio,34
3,1,Loja,alto,61
4,2,Ambato,medio,45


In [131]:
# Clasificación manual según la naturaleza de la variable
numeric_features = ['edad']
categorical_nominal_features = ['sexo', 'ciudad']
categorical_ordinal_features = ['colesterol']  # Orden lógico: bajo < medio < alto < muy alto

# Impresión de la clasificación requerida
print("--- Clasificación de Variables ---")
print("Variables numéricas:", numeric_features)
print("Variables categóricas nominales:", categorical_nominal_features)
print("Variables categóricas ordinales:", categorical_ordinal_features)
print("Variable objetivo (target): ['diabetes']")


--- Clasificación de Variables ---
Variables numéricas: ['edad']
Variables categóricas nominales: ['sexo', 'ciudad']
Variables categóricas ordinales: ['colesterol']
Variable objetivo (target): ['diabetes']


<div id="analisis-variables" style="color:#106ba3"><h3>Análisis de variables categóricas</h3> </div>

In [132]:
def analisisVariables(dataframe, categorical_nominal_features):
    cantidadTotalVariables = len(dataframe.columns)
    print("Cantidad de variables antes de transformación: ", cantidadTotalVariables)
    
    cantidadVariablesNominales = len(categorical_nominal_features)
    cantidadVariablesBinarias = 0
    
    for variable in categorical_nominal_features:
        cantidadCategorias = dataframe[variable].nunique()
        cantidadVariablesBinarias += cantidadCategorias
        print(f"Cantidad de categorías en la variable categórica nominal '{variable}': {cantidadCategorias}")
        
    print("Cantidad de variables binarias (One-Hot) que reemplazarán a las nominales: ", cantidadVariablesBinarias)
    
    # Cálculo matemático del cambio dimensional: Dim_final = Dim_actual - Nominales + Binarias
    cantidadTotalVariablesConTransformacion = cantidadTotalVariables - cantidadVariablesNominales + cantidadVariablesBinarias
    return cantidadTotalVariablesConTransformacion

# Ejecución de la función con los parámetros definidos
cantidadTotalVariablesConTransformacion = analisisVariables(dataframe, categorical_nominal_features)

print("Cantidad de variables que habrá después de la transformación: ", cantidadTotalVariablesConTransformacion)

Cantidad de variables antes de transformación:  4
Cantidad de categorías en la variable categórica nominal 'sexo': 2
Cantidad de categorías en la variable categórica nominal 'ciudad': 6
Cantidad de variables binarias (One-Hot) que reemplazarán a las nominales:  8
Cantidad de variables que habrá después de la transformación:  10


Hay 2 variables categóricas nominales: 'sexo' y 'ciudad'. Estas 2 variables categóricas nominales deben ser reemplazadas por variables binarias.

**sexo:** En el dataset se presentan 2 categorías, que darán lugar a 2 variables binarias por el proceso de transformación binaria.
- 1 : Masculino / Identificador 1
- 2 : Femenino / Identificador 2

**ciudad:** En el dataset se presentan 6 categorías (ciudades ecuatorianas), que darán lugar a 6 variables binarias por el proceso de transformación binaria.
- Cuenca
- Quito
- Guayaquil
- Loja
- Ambato
- Machala

cantidadVariablesBinarias = 2 + 6 = 8 variables binarias en total

Las 2 variables categóricas nominales serán reemplazadas por las 8 variables binarias.

Por lo tanto,

A las 4 variables iniciales (excluyendo la variable objetivo 'diabetes') se eliminarán las 2 variables categóricas nominales y se agregarán las 8 variables binarias.

4 - 2 + 8 = 10 variables

El dataframe luego de la transformación categórica a numérica tendrá 10 variables independientes en total.

<div id="transform" style="color:#37475a; border-bottom: 7px solid orange; width: 100%; margin-bottom: 15px; padding-bottom: 2px"><h2>3. Realizar transformación categórica</h2> </div>

In [133]:
# 1. Definición de las variables por tipo
categorical_ordinal_features = ["colesterol"]
categorical_nominal_features = ["sexo", "ciudad"]
numeric_features = ["edad"]

# 2. Configuración del mapeo explícito para la variable ordinal
# Orden establecido: bajo (0) < medio (1) < alto (2) < muy alto (3)
orden_colesterol = [["bajo", "medio", "alto", "muy alto"]]
ordinal_transformer = OrdinalEncoder(categories=orden_colesterol)

# 3. Configuración del codificador One-Hot para nominales
onehot_transformer = OneHotEncoder(sparse_output=False, handle_unknown="ignore")

# 4. Construcción del ColumnTransformer
preprocesorCategorico = ColumnTransformer(
    transformers=[
        ("catord", ordinal_transformer, categorical_ordinal_features),
        ("catnom", onehot_transformer, categorical_nominal_features),
    ],
    remainder="passthrough",  # Mantiene 'edad' al final del array resultante
    n_jobs=-1,
)

print("Pipeline de preprocesamiento configurado con éxito.")

Pipeline de preprocesamiento configurado con éxito.


In [134]:
# 5. Ejecución de la transformación en las variables independientes (X)
dataframeTransformado = copy.deepcopy(dataframe)
X_Transformado = preprocesorCategorico.fit_transform(dataframeTransformado)

# Inspección de la matriz resultante (NumPy array)
print("Dimensiones de la matriz X_Transformado:", X_Transformado.shape)
print("\nPrimeros 3 registros de la matriz generada:")
print(X_Transformado[:3])

Dimensiones de la matriz X_Transformado: (6, 10)

Primeros 3 registros de la matriz generada:
[[ 0.  1.  0.  0.  1.  0.  0.  0.  0. 18.]
 [ 2.  0.  1.  0.  0.  0.  0.  0.  1. 52.]
 [ 1.  0.  1.  0.  0.  1.  0.  0.  0. 34.]]


In [135]:
# 6. Reconstrucción de los nombres de las columnas
cnamesDataset = []

# Primero entran las ordinales
cnamesDataset.extend(categorical_ordinal_features)

# Segundo, las nominales codificadas (OneHot)
cnamesNominales = preprocesorCategorico.named_transformers_["catnom"].get_feature_names_out(categorical_nominal_features)
cnamesDataset.extend(cnamesNominales)

# Al final, las variables del 'remainder' (numéricas)
cnamesDataset.extend(numeric_features)

# Creación del DataFrame intermedio de predictores
dataframeFinal = pd.DataFrame(data=X_Transformado, columns=cnamesDataset)

# Verificación de las características transformadas antes de añadir el target
dataframeFinal.head(6)

,colesterol,sexo_1,sexo_2,ciudad_Ambato,ciudad_Cuenca,ciudad_Guayaquil,ciudad_Loja,ciudad_Machala,ciudad_Quito,edad
0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,18.0
1,2.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,52.0
2,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,34.0
3,2.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,61.0
4,1.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,45.0
5,3.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,67.0


In [136]:
# 7. Integración de la variable objetivo binarizada (Y)
Y_binarizado = Y.replace({"si": 1, "no": 0})

# Concatenación horizontal reseteando índices para evitar desalineación
dataframeFinal = pd.concat([dataframeFinal, Y_binarizado.reset_index(drop=True)], axis=1)

print(f"Dimensiones finales del dataset: {dataframeFinal.shape}")

# Inspección final del dataset procesado y listo para el modelo
dataframeFinal.head(6)

Dimensiones finales del dataset: (6, 11)


,colesterol,sexo_1,sexo_2,ciudad_Ambato,ciudad_Cuenca,ciudad_Guayaquil,ciudad_Loja,ciudad_Machala,ciudad_Quito,edad,diabetes
0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,18.0,0
1,2.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,52.0,1
2,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,34.0,0
3,2.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,61.0,1
4,1.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,45.0,0
5,3.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,67.0,1


<div id="std" style="color:#37475a; border-bottom: 7px solid orange; width: 100%; margin-bottom: 15px; padding-bottom: 2px"><h2>4. Aplicar estandarización</h2> </div>

In [137]:
# 1. Separar variables predictoras (X) y variable objetivo (Y)
X_final = dataframeFinal.drop(["diabetes"], axis=1)
Y_final = dataframeFinal["diabetes"]

# 2. Configurar y aplicar la estandarización con StandardScaler
scaler = StandardScaler()
X_estandarizado = scaler.fit_transform(X_final)

# 3. Reconstruir el DataFrame con los datos escalados y la variable objetivo
dataframeEstandarizado = pd.DataFrame(data=X_estandarizado, columns=X_final.columns)
dataframeEstandarizado = pd.concat([dataframeEstandarizado, Y_final.reset_index(drop=True)], axis=1)

dataframeEstandarizado.head(6)

,colesterol,sexo_1,sexo_2,ciudad_Ambato,ciudad_Cuenca,ciudad_Guayaquil,ciudad_Loja,ciudad_Machala,ciudad_Quito,edad,diabetes
0,-1.566699,1.0,-1.0,-0.447214,2.236068,-0.447214,-0.447214,-0.447214,-0.447214,-1.708466,0
1,0.522233,-1.0,1.0,-0.447214,-0.447214,-0.447214,-0.447214,-0.447214,2.236068,0.353824,1
2,-0.522233,-1.0,1.0,-0.447214,-0.447214,2.236068,-0.447214,-0.447214,-0.447214,-0.737976,0
3,0.522233,1.0,-1.0,-0.447214,-0.447214,-0.447214,2.236068,-0.447214,-0.447214,0.899725,1
4,-0.522233,-1.0,1.0,2.236068,-0.447214,-0.447214,-0.447214,-0.447214,-0.447214,-0.070765,0
5,1.566699,1.0,-1.0,-0.447214,-0.447214,-0.447214,-0.447214,2.236068,-0.447214,1.263658,1


### Importancia de la Estandarización en el algoritmo KNN

El algoritmo K-Nearest Neighbors (KNN) clasifica los nuevos datos basándose en la distancia (generalmente la distancia euclídea) hacia sus vecinos más cercanos en el espacio de características. Debido a esta naturaleza geométrica, la estandarización es un paso crítico por las siguientes razones:

1. **Sensibilidad a la escala:** Si las variables tienen rangos numéricos muy diferentes (por ejemplo, la `edad` que varía entre 18 y 67 años frente a una variable binaria como `sexo_1` que solo toma valores de 0 o 1), la distancia euclídea se verá dominada casi en su totalidad por la variable con el rango numérico más grande.
2. **Ponderación equitativa:** Al aplicar `StandardScaler`, cada característica se transforma para tener una media ($\mu = 0$) y una desviación estándar ($\sigma = 1$). Esto sitúa a todas las variables predictoras en la misma escala, garantizando que cada atributo aporte de manera equitativa al cálculo de la distancia.
3. **Evitar sesgos en la predicción:** Sin el escalado, el modelo asumiría erróneamente que los cambios numéricos en la variable de mayor magnitud son más significativos para predecir la diabetes que los cambios en las variables de menor escala, degradando la precisión del clasificador.

<div id="knn" style="color:#37475a; border-bottom: 7px solid orange; width: 100%; margin-bottom: 15px; padding-bottom: 2px"><h2>5. Implementar KNN con distancia euclídea y k=3</h2> </div>

In [138]:
X_historico = dataframeEstandarizado.drop(["diabetes"], axis=1).values
Y_historico = dataframeEstandarizado["diabetes"].values


def evaluar_nuevo_paciente(sexo, ciudad, colesterol, edad):
    cnames = []
    cnames.extend(categorical_ordinal_features)
    
    cnamesNominales = preprocesorCategorico.named_transformers_["catnom"].get_feature_names_out(categorical_nominal_features)
    cnames.extend(cnamesNominales)
    
    cnames.extend(numeric_features)

    # 2. Creación del DataFrame para el nuevo paciente
    nuevo_paciente_df = pd.DataFrame(
        [[sexo, ciudad, colesterol, edad]],
        columns=["sexo", "ciudad", "colesterol", "edad"],
    )

    print("1. Datos del Nuevo Paciente Registrado")
    display(nuevo_paciente_df)

    # 3. Aplicación de las mismas transformaciones del preprocesador
    nuevo_transformado = preprocesorCategorico.transform(nuevo_paciente_df)
    
    # Se asignan los nombres limpios idénticos a los del set de entrenamiento
    nuevo_transformado_df = pd.DataFrame(nuevo_transformado, columns=cnames)

    print("\n2. Datos tras Codificación (ColumnTransformer)")
    display(nuevo_transformado_df)

    # 4. Aplicación de la estandarización con el StandardScaler entrenado
    # Ahora las columnas coinciden exactamente con las de entrenamiento
    nuevo_estandarizado = scaler.transform(nuevo_transformado_df)
    nuevo_estandarizado_df = pd.DataFrame(
        nuevo_estandarizado, columns=cnames
    )

    print("\n3. Vector Final Estandarizado (Listo para Medir Distancia)")
    display(nuevo_estandarizado_df)

    vector_nuevo = nuevo_estandarizado[0]

    # 5. Cálculo manual de la distancia euclídea respecto a todos los pacientes
    distancias = []
    for i in range(len(X_historico)):
        distancia = np.sqrt(np.sum((X_historico[i] - vector_nuevo) ** 2))
        distancias.append((distancia, Y_historico[i], i))

    # 6. Selección de los k=3 vecinos más cercanos
    distancias.sort(key=lambda item: item[0])
    vecinos_seleccionados = distancias[:3]

    datos_vecinos = []
    for idx, (dist, target, original_idx) in enumerate(
        vecinos_seleccionados, 1
    ):
        estado_diabetes = "si" if target == 1 else "no"
        datos_vecinos.append(
            {
                "Prioridad Vecino": f"Vecino {idx}",
                "Índice Fila Original": original_idx,
                "Distancia Euclídea": round(dist, 4),
                "Diabetes (Clase Real)": estado_diabetes,
            }
        )
    vecinos_df = pd.DataFrame(datos_vecinos)

    print("\n4. Análisis de Vecinos Más Cercanos (k=3)")
    display(vecinos_df)

    # 7. Votación mayoritaria manual (k=3)
    votos_diabetes = [vecino[1] for vecino in vecinos_seleccionados]
    conteo_si = votos_diabetes.count(1)
    conteo_no = votos_diabetes.count(0)

    prediccion_final = "si" if conteo_si > conteo_no else "no"

    print("\n5. Predicción y Diagnóstico Final")
    print(f"Votos para 'si': {conteo_si}")
    print(f"Votos para 'no': {conteo_no}")
    print(f"Resultado del diagnóstico: El paciente {prediccion_final} tiene diabetes.")

print("Función de evaluación para nuevo paciente definida con éxito.")

Función de evaluación para nuevo paciente definida con éxito.


<div id="ejemplo" style="color:#37475a; border-bottom: 7px solid orange; width: 100%; margin-bottom: 15px; padding-bottom: 2px"><h2>Ejemplo de prueba</h2> </div>

In [139]:
evaluar_nuevo_paciente(
    sexo=1,           # OPCIONES: 1 = Femenino, 2 = Masculino
    ciudad="Cuenca",  # OPCIONES: "Ambato", "Cuenca", "Guayaquil", "Loja", "Machala", "Quito" (Respetar mayúsculas)
    colesterol="medio",# OPCIONES: "bajo", "medio", "alto", "muy alto"
    edad=55           # OPCIONES: Valor numérico entero (Ej. 18 a 100) que representa los años del paciente
)

1. Datos del Nuevo Paciente Registrado


,sexo,ciudad,colesterol,edad
0,1,Cuenca,medio,55



2. Datos tras Codificación (ColumnTransformer)


,colesterol,sexo_1,sexo_2,ciudad_Ambato,ciudad_Cuenca,ciudad_Guayaquil,ciudad_Loja,ciudad_Machala,ciudad_Quito,edad
0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,55.0



3. Vector Final Estandarizado (Listo para Medir Distancia)


,colesterol,sexo_1,sexo_2,ciudad_Ambato,ciudad_Cuenca,ciudad_Guayaquil,ciudad_Loja,ciudad_Machala,ciudad_Quito,edad
0,-0.522233,1.0,-1.0,-0.447214,2.236068,-0.447214,-0.447214,-0.447214,-0.447214,0.535791



4. Análisis de Vecinos Más Cercanos (k=3)


,Prioridad Vecino,Índice Fila Original,Distancia Euclídea,Diabetes (Clase Real)
0,Vecino 1,0,2.4754,no
1,Vecino 2,3,3.9526,si
2,Vecino 3,5,4.3924,si



5. Predicción y Diagnóstico Final
Votos para 'si': 2
Votos para 'no': 1
Resultado del diagnóstico: El paciente si tiene diabetes.


<div id="conclusiones" style="color:#37475a; border-bottom: 7px solid orange; width: 100%; margin-bottom: 15px; padding-bottom: 2px"><h2>Conclusiones</h2> </div>

- **Transformación de Variables en Salud:** La práctica se centra en la transformación adaptada de variables clínicas y demográficas de pacientes ecuatorianos. Esto incluye la codificación ordinal estructurada para niveles de riesgo (`colesterol`) y codificación One-Hot para variables cualitativas (`sexo` y `ciudad`), permitiendo que el algoritmo interprete correctamente datos médicos heterogéneos.

- **Estructuración del Preprocesamiento:** Se implementa un flujo de trabajo claro empleando `ColumnTransformer` para separar y ejecutar de forma automatizada las transformaciones según la naturaleza de cada atributo. Esto garantiza la reproducibilidad en la preparación de matrices de datos de salud y previene el error de fuga de datos (*data leakage*).

- **Clasificación Manual con KNN:** A diferencia de las librerías automatizadas, se implementa el algoritmo K-Nearest Neighbors (KNN) de forma manual utilizando lógica vectorial. El modelo calcula directamente la proximidad geométrica en un espacio multidimensional para clasificar a los pacientes según la presencia o ausencia de diabetes.

- **Predicción Basada en Votación Mayoritaria:** El sistema determina el diagnóstico predictivo final mediante un mecanismo de votación mayoritaria estricta entre los $k=3$ vecinos más cercanos. Esto ejemplifica de manera transparente los fundamentos teóricos de la clasificación supervisada no paramétrica.

- **Estandarización Crítica:** Se aplica estandarización estadística (`StandardScaler`) a las variables predictoras para lograr una media cero y varianza unitaria. Esta fase es indispensable para mitigar sesgos de magnitud, evitando que variables con rangos más amplios (como la `edad`) eclipsen de forma artificial el peso de los indicadores binarios en el cálculo de distancias.

- **Sensibilidad a la Distancia Euclídea:** La práctica demuestra de manera matemática cómo la distancia euclídea es altamente sensible a las escalas de medición. Ajustar la métrica de distancia tras un correcto escalado asegura que cada factor demográfico y de salud aporte de manera equitativa a la detección de similitudes entre pacientes.

- **Diagnóstico Estructurado de Nuevas Instancias:** El diseño culmina en una función predictiva integrada que replica las transformaciones históricas sobre datos de un nuevo paciente en tiempo real. Este enfoque estructurado emula el despliegue real de un sistema de soporte a la decisión clínica en el entorno biomédico.

Esta práctica proporciona una sólida comprensión de cómo realizar ingeniería de características en variables médicas e implementar la lógica nativa del modelo KNN para la clasificación diagnóstica con base en métricas de proximidad.

<div id="referencias" style="color:#37475a; border-bottom: 7px solid orange; width: 100%; margin-bottom: 15px; padding-bottom: 2px"><h2>Referencias</h2> </div>

* **Hurtado, R. (2026).** *Transformaciones y Clasificación KNN [Notebook de Jupyter]*. Recuperado de GitHub: [Machine Learning Engineering & Production Portfolio](https://github.com/Remigiohurtadoins/MachineLearning-Engineering-Production-Portfolio/blob/main/2.Data%20Preparation%20ML%20Loan%20Approval%20Financial%20Institution/Notebook/TransformacionesyClasificacionKNN.ipynb)